In [2]:
import dash
from dash import dcc, html, Input, Output, State, callback
import dash_table
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

C:\Users\MEGH BAVARVA\AppData\Local\Temp\ipykernel_14808\1029087819.py:3: UserWarning: 
The dash_table package is deprecated. Please replace
`import dash_table` with `from dash import dash_table`

Also, if you're using any of the table format helpers (e.g. Group), replace 
`from dash_table.Format import Group` with 
`from dash.dash_table.Format import Group`
  import dash_table


In [3]:
df=pd.read_csv('data/gandhinagar_property_apartments_recommender_ready.csv')  # Load your dataset here
df.head()

,location,price_inr_in_lakhs,area_sqft,description,property_url,bathrooms,balconies,current_floor,total_floors,furnishing_status,mapped_area,facing,property_age,property_type,bedrooms,area_type,property_status
0,"Sargasan, Gandhinagar",126.00,2916.0,Experience a new style of living with Altezza ...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,1.0,4.0,8.0,Unknown,Sargasan,Unknown,New,apartment,3.0,Super Built-up,Ready_to_Move
1,"Pethapur, Gandhinagar",51.99,1755.0,Experience a new style of living with Satyamev...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,2.0,3.0,7.0,Unknown,Pethapur,Unknown,New,apartment,3.0,Super Built-up,Under_Construction
2,"Raysan, Gandhinagar",97.00,1908.0,This is your chance to a 3 bhk apartment / fla...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,2.0,4.0,8.0,Unknown,Raysan,Unknown,New,apartment,3.0,Super Built-up,Ready_to_Move
3,"Raysan, Gandhinagar",125.00,2205.0,We are the proud owners of this 3 bhk apartmen...,https://www.99acres.com/3-bhk-bedroom-apartmen...,2.0,2.0,9.0,13.0,Unknown,Raysan,Unknown,New,apartment,3.0,Carpet,Ready_to_Move
4,"Urjanagar 1, Randesan, Gandhinagar",79.00,1755.0,Situated on the top floor (8th) of a mid-Rise ...,https://www.99acres.com/3-bhk-bedroom-apartmen...,3.0,1.0,8.0,13.0,unfurnished,Randesan,Unknown,5-10 years,apartment,3.0,Carpet,Ready_to_Move


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1304 entries, 0 to 1303
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   location            1304 non-null   str    
 1   price_inr_in_lakhs  1304 non-null   float64
 2   area_sqft           1304 non-null   float64
 3   description         1203 non-null   str    
 4   property_url        1304 non-null   str    
 5   bathrooms           1304 non-null   float64
 6   balconies           1304 non-null   float64
 7   current_floor       1304 non-null   float64
 8   total_floors        1304 non-null   float64
 9   furnishing_status   1304 non-null   str    
 10  mapped_area         1304 non-null   str    
 11  facing              1304 non-null   str    
 12  property_age        1304 non-null   str    
 13  property_type       1304 non-null   str    
 14  bedrooms            1304 non-null   float64
 15  area_type           1304 non-null   str    
 16  property_status  

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [6]:
df[['price_inr_in_lakhs']]

,price_inr_in_lakhs
0,126.00
1,51.99
2,97.00
3,125.00
4,79.00
...,...
1299,40.00
1300,32.00
1301,90.00
1302,78.00


In [7]:
def recommend_by_price(selected_price, top_n=5):

    temp = df.copy()

    temp['price_diff'] = abs(
        temp['price_inr_in_lakhs'] - selected_price
    )

    recommendations = (
        temp.sort_values('price_diff')
            .head(top_n)
    )

    return recommendations[[
        'location',
        'price_inr_in_lakhs',
        'area_sqft',
        'bedrooms',
        'property_url'
    ]]

In [8]:
selected_price = 120  # Lakhs

In [9]:
recommend_by_price(120)

,location,price_inr_in_lakhs,area_sqft,bedrooms,property_url
43,"Sargasan, Gandhinagar",120.0,2313.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-vinayak-sapphire-sargasan-gandhinagar-2313-sqft-spid-J89832344
381,"Kudasan, Gandhinagar",120.0,2925.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-kudasan-gandhinagar-2925-sqft-spid-N89951698
1168,"Raysan, Gandhinagar",120.0,2520.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-vinayak-skydeck-raysan-gandhinagar-2520-sqft-spid-T86369204
1110,"Kudasan, Gandhinagar",120.0,2475.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-vision-ventilla-kudasan-gandhinagar-2475-sqft-spid-C87906722
304,"Kudasan, Gandhinagar",120.0,2376.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-kudasan-gandhinagar-2376-sqft-r1-spid-F69699670


In [12]:
def recommend_by_area(selected_area,
                      area_type=None,
                      top_n=5):

    temp = df.copy()

    # Filter by area type if provided
    if area_type is not None:
        temp = temp[temp['area_type'] == area_type]

    # Calculate area difference
    temp['area_diff'] = abs(
        temp['area_sqft'] - selected_area
    )

    recommendations = (
        temp.sort_values('area_diff')
            .head(top_n)
    )

    return recommendations[[
        'location',
        'area_sqft',
        'area_type',
        'price_inr_in_lakhs',
        'bedrooms',
        'property_url'
    ]]

In [16]:
recommend_by_area(
    selected_area=1200,
    top_n=5
)

,location,area_sqft,area_type,price_inr_in_lakhs,bedrooms,property_url
843,Swagat RaForest 4,1200.0,Carpet,45.0,2.0,https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-swagat-rain-forest-4-sargasan-gandhinagar-1200-sqft-spid-N90525252
876,"Chiloda, Gandhinagar",1200.0,Carpet,38.0,2.0,https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-psy-pramukh-orchid-chiloda-gandhinagar-1200-sqft-r3-spid-L68555626
323,"Koba, Gandhinagar",1200.0,Carpet,55.0,2.0,https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-psy-pramukh-elegance-koba-gandhinagar-1200-sqft-r1-spid-A78785439
1154,"Sector 22, Gandhinagar",1200.0,Super Built-up,70.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-harihar-apartment-sector-22-gandhinagar-1200-sqft-spid-N87598520
875,"Sargasan, Gandhinagar",1200.0,Super Built-up,49.0,2.0,https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-psy-pramukh-elysium-sargasan-gandhinagar-1200-sqft-r1-spid-Q88749600


In [17]:
df.columns

Index(['location', 'price_inr_in_lakhs', 'area_sqft', 'description',
       'property_url', 'bathrooms', 'balconies', 'current_floor',
       'total_floors', 'furnishing_status', 'mapped_area', 'facing',
       'property_age', 'property_type', 'bedrooms', 'area_type',
       'property_status'],
      dtype='str')

In [18]:
def bedroom_similarity(df, selected_bedroom):

    temp = df.copy()

    temp['bedroom_score'] = (
        1 - abs(temp['bedrooms'] - selected_bedroom)
              / temp['bedrooms'].max()
    )

    return temp['bedroom_score']

In [19]:
bedroom_similarity(df, selected_bedroom=3)

0       1.000000
1       1.000000
2       1.000000
3       1.000000
4       1.000000
          ...   
1299    0.833333
1300    0.833333
1301    1.000000
1302    1.000000
1303    0.833333
Name: bedroom_score, Length: 1304, dtype: float64

In [ ]:
numeric_features = [
    'price_inr_in_lakhs',
    'area_sqft',
    'bedrooms',
    'bathrooms',
    'balconies',
    'current_floor',
    'total_floors'
]

categorical_features = [
    'mapped_area',
    'property_age',
    'property_type',
    'furnishing_status',
    'facing',
    'area_type'
]

In [22]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

num_matrix = scaler.fit_transform(
    df[numeric_features]
)

In [23]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

cat_matrix = encoder.fit_transform(
    df[categorical_features]
)

In [ ]:
num_weights = {
    'price_inr_in_lakhs': 0.15,
    'area_sqft': 0.15,
    'bedrooms': 0.12,
    'bathrooms': 0.05,
    'balconies': 0.03,
    'current_floor': 0.01,
    'total_floors': 0.01
}

for i, col in enumerate(numeric_features):
    num_matrix[:, i] *= num_weights[col]

In [ ]:
cat_weights = {
    'mapped_area': 0.20,
    'area_type':0.10,
    'property_age': 0.05,
    'property_type': 0.10,
    'furnishing_status': 0.02,
    'facing': 0.01
}

In [26]:
start = 0

for feature in categorical_features:

    n_cols = len(
        encoder.categories_[
            categorical_features.index(feature)
        ]
    )

    cat_matrix[:, start:start+n_cols] *= (
        cat_weights[feature] / n_cols
    )

    start += n_cols

In [27]:
import numpy as np

X = np.hstack([
    num_matrix,
    cat_matrix
])

In [28]:
property_idx = 25

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_properties(
    property_idx,
    X,
    df,
    top_n=5
):

    similarity_scores = cosine_similarity(
        X[property_idx].reshape(1,-1),
        X
    )[0]

    similar_idx = (
        similarity_scores
        .argsort()[::-1]
    )

    similar_idx = [
        i for i in similar_idx
        if i != property_idx
    ][:top_n]

    return df.iloc[similar_idx][[
        'Mapped_Area',
        'price_inr_in_lakhs',
        'area_sqft',
        'bedrooms',
        'property_url'
    ]]

In [30]:
df.loc[25]

location                                                                                                                                                                                                                                                                                     Kudasan, Gandhinagar
price_inr_in_lakhs                                                                                                                                                                                                                                                                                          145.0
area_sqft                                                                                                                                                                                                                                                                                                  3230.0
description           Shivalay parisar is a well-Maintained residential apartment 

In [31]:
recommend_properties(
    property_idx=25,
    X=X,
    df=df,
    top_n=5
)

,location,price_inr_in_lakhs,area_sqft,bedrooms,property_url
1140,"Kudasan, Gandhinagar",146.0,3060.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-the-grasslands-kudasan-gandhinagar-3060-sqft-spid-T87708956
328,"Kudasan, Gandhinagar",140.0,2669.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-kudasan-gandhinagar-2669-sqft-spid-D90359798
256,"Kudasan, Gandhinagar",140.0,2475.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-revanta-fortune-3-kudasan-gandhinagar-2475-sqft-spid-Y88776416
1110,"Kudasan, Gandhinagar",120.0,2475.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-vision-ventilla-kudasan-gandhinagar-2475-sqft-spid-C87906722
401,"Kudasan, Gandhinagar",116.0,2610.0,3.0,https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-shree-sarvatra-aryan-imperial-kudasan-gandhinagar-2610-sqft-r1-spid-M89375341
